Домашнее задание
Две части:
1. Задача с матчингом таймстемпов.
Допустим, есть две видеокамеры, снимающие одну и ту же сцену, и два массива таймстемпов для каждого кадра этих камер. Надо для каждого таймстемпа первой камеры найти индекс ближайшего таймстемпа второй камеры (описание приложил)
2. Задача со стримом медианы.
Дан массив размера n и длина окна k. Надо найти медиану в скользящем окне.
Пример: nums = np.array([1,3,-1,-3,5,3,6,7]), k = 3 => ответ [1,-1,-1,3,5,6]

Критерии оценивания:
Задачи с разным весом. За первую задачу максимальная оценка 7 баллов, за вторую - 3 балла.
Обе задачи должны быть оптимальны как алгоритмически, так и с точки зрения имплементации. Для достижения оптимальной имплементации рекомендуется вспомнить материалы первого занятия.
Более подробное описание можно найти в приложенном файле.[Текст ссылки](https://)

In [1]:
import time
import numpy as np
from heapq import heappush, heappop

def match_timestamps(timestamps1: np.ndarray, timestamps2: np.ndarray) -> np.ndarray:
    """
    Timestamp matching function. It returns such array `matching` of length len(timestamps1),
    that for each index i of timestamps1 the element matching[i] contains
    the index j of timestamps2, so that the difference between
    timestamps2[j] and timestamps1[i] is minimal.
    Example:
        timestamps1 = [0, 0.091, 0.5]
        timestamps2 = [0.001, 0.09, 0.12, 0.6]
        => matching = [0, 1, 3]
    """
    if len(timestamps2) == 0:
        return np.empty(0, dtype=np.uint32)

    idx = np.searchsorted(timestamps2, timestamps1, side='left')
    best_idx = idx.copy()

    left = idx - 1
    valid_left = left >= 0
    valid_right = idx < len(timestamps2)
    both = valid_left & valid_right

    if np.any(both):
        t1 = timestamps1[both]
        diff_left = np.abs(t1 - timestamps2[left[both]])
        diff_right = np.abs(t1 - timestamps2[idx[both]])
        use_left = diff_left <= diff_right
        best_idx[both] = np.where(use_left, left[both], idx[both])

    only_left = valid_left & ~valid_right
    if np.any(only_left):
        best_idx[only_left] = left[only_left]

    return best_idx.astype(np.uint32)


def sliding_window_median(nums, k):
    """
    Находит медиану в скользящем окне размера k.
    Медиана — это средний элемент. Если разделить числа на две половины (меньшие и большие), медиана будет между ними. Делаем 2 кучи: max_heap для меньших и min_heap для больших.
    Когда добавляем число, помещаем его в соответствующую кучу. Если кучи не сбалансированы, перемещаем элементы между ними. Когда удаляем число, помечаем его для ленивого удаления и балансируем кучи.
    Получаем медиану из корней куч. Если кучи равны по размеру, берем среднее из корней, иначе — корень max_heap.
    Сложность: O(n log k)
    """
    if len(nums) < k:
        return np.array([])

    max_heap = []  # Макс-куча (хранятся отрицания) для левой половины
    min_heap = []  # Мин-куча для правой половины
    to_remove = {}  # Словарь для ленивого удаления: значение -> количество помеченных на удаление
    max_size = min_size = 0  # Логические размеры куч (без учёта помеченных на удаление)

    def add(num):
        nonlocal max_size, min_size
        # Определяем, в какую кучу добавить новое число
        if not max_heap or num <= -max_heap[0]:
            heappush(max_heap, -num)
            max_size += 1
        else:
            heappush(min_heap, num)
            min_size += 1
        balance()

    def remove(num):
        nonlocal max_size, min_size
        # Помечаем число на удаление
        to_remove[num] = to_remove.get(num, 0) + 1
        # Уменьшаем логический размер соответствующей кучи
        if num <= -max_heap[0]:
            max_size -= 1
        else:
            min_size -= 1
        balance()

    def balance():
        nonlocal max_size, min_size
        clean()
        # Перемещаем элементы, чтобы поддерживать баланс: len(max_heap) >= len(min_heap)
        if max_size > min_size + 1:
            val = -heappop(max_heap)
            max_size -= 1
            heappush(min_heap, val)
            min_size += 1
        elif min_size > max_size:
            val = heappop(min_heap)
            min_size -= 1
            heappush(max_heap, -val)
            max_size += 1
        clean()

    def clean():
        # Удаляем верхние элементы куч, если они помечены на удаление
        while max_heap and to_remove.get(-max_heap[0], 0) > 0:
            to_remove[-max_heap[0]] -= 1
            heappop(max_heap)
        while min_heap and to_remove.get(min_heap[0], 0) > 0:
            to_remove[min_heap[0]] -= 1
            heappop(min_heap)

    def get_median():
        clean()
        # Если окно нечётное — медиана в max_heap, иначе — среднее двух корней
        if max_size == min_size:
            return (-max_heap[0] + min_heap[0]) / 2
        else:
            return -max_heap[0]

    result = []
    # Инициализация первого окна
    for i in range(k):
        add(nums[i])
    result.append(get_median())

    # Скользящее окно: удаляем старый элемент, добавляем новый
    for i in range(k, len(nums)):
        remove(nums[i - k])
        add(nums[i])
        result.append(get_median())

    return np.array(result)


def make_timestamps(fps: int, st_ts: float, fn_ts: float) -> np.ndarray:
    """
    Create array of timestamps. This array is discretized with fps,
    but not evenly.
    Timestamps are assumed sorted nad unique.
    Parameters:
    - fps: int
        Average frame per second
    - st_ts: float
        First timestamp in the sequence
    - fn_ts: float
        Last timestamp in the sequence
    Returns:
        np.ndarray: synthetic timestamps
    """
    # генерация равномерных таймстемпов
    timestamps = np.linspace(st_ts, fn_ts, int((fn_ts - st_ts) * fps))
    # добавление шума, симулирующего джиттер
    timestamps += np.random.randn(len(timestamps))
    # сортировка и удаление дубликатов
    timestamps = np.unique(np.sort(timestamps))
    return timestamps


def test_match_timestamps():
    """Тест для задачи сопоставления таймстемпов двух камер."""
    print("=== Тест задачи 1: сопоставление таймстемпов ===")
    current_time = time.time()
    timestamps1 = make_timestamps(30, current_time - 100, current_time + 3600 * 2)
    timestamps2 = make_timestamps(60, current_time + 200, current_time + 3600 * 2.5)

    print(f"Камера 1: {len(timestamps1)} кадров")
    print(f"Камера 2: {len(timestamps2)} кадров")

    # Замер времени выполнения сопоставления
    start_time = time.perf_counter()
    matching = match_timestamps(timestamps1, timestamps2)
    elapsed = time.perf_counter() - start_time

    # Вывод первых 2 совпадений
    print("\nПервые 2 сопоставления:")
    for i in range(min(2, len(matching))):
        j = int(matching[i])
        diff = abs(timestamps1[i] - timestamps2[j])
        print(f"  Кадр {i} (камера1: {timestamps1[i]:.3f}) → кадр {j} (камера2: {timestamps2[j]:.3f}), разница: {diff:.6f} с")


def test_sliding_window_median():
    """Тест для задачи скользящей медианы."""
    print("\n=== Тест задачи 2: скользящая медиана ===")

    # Базовый пример из условия задачи
    print("1. Базовый пример из условия:")
    nums1 = np.array([1, 3, -1, -3, 5, 3, 6, 7])
    k1 = 3
    result1 = sliding_window_median(nums1, k1)
    expected = np.array([1.0, -1.0, -1.0, 3.0, 5.0, 6.0])
    print(f"  Вход: nums = {nums1}, k = {k1}")
    print(f"  Результат: {result1}")
    print(f"  Ожидаемо:  {expected}\n")

    # Тест производительности: 1_000_000 элементов, окно 1000
    print("2. Тест производительности (1 000 000 элементов, окно 1000):")
    np.random.seed(42)
    large_nums = np.random.randint(-1000, 1000, size=1_000_000)
    k2 = 1000

    start = time.perf_counter()
    result2 = sliding_window_median(large_nums, k2)
    elapsed = time.perf_counter() - start

    print(f"  Время выполнения: {elapsed:.3f} секунд")
    print(f"  Длина результата: {len(result2)}")
    print(f"  Первые 5 медиан: {result2[:5]}")


if __name__ == '__main__':
    test_match_timestamps()
    test_sliding_window_median()


=== Тест задачи 1: сопоставление таймстемпов ===
Камера 1: 218998 кадров
Камера 2: 527995 кадров

Первые 2 сопоставления:
  Кадр 0 (камера1: 1772832013.096) → кадр 0 (камера2: 1772832314.120), разница: 301.023989 с
  Кадр 1 (камера1: 1772832014.205) → кадр 0 (камера2: 1772832314.120), разница: 299.915244 с

=== Тест задачи 2: скользящая медиана ===
1. Базовый пример из условия:
  Вход: nums = [ 1  3 -1 -3  5  3  6  7], k = 3
  Результат: [ 1 -1 -1  3  5  6]
  Ожидаемо:  [ 1. -1. -1.  3.  5.  6.]

2. Тест производительности (1 000 000 элементов, окно 1000):
  Время выполнения: 9.789 секунд
  Длина результата: 999001
  Первые 5 медиан: [56.  56.  53.5 53.5 51. ]
